In [1]:
install.packages("xgboost")


下载的二进制程序包在
	/var/folders/1h/1p8bcks947q878b7gn_8jhtm0000gn/T//Rtmp2BtSHl/downloaded_packages里


In [2]:
library(xgboost)

In [5]:
packageVersion("xgboost")

[1] ‘3.2.1.1’

In [7]:
library(xgboost)

# Load data
load_reg_train <- read.csv("regression_train.csv")
test <- read.csv("regression_test.csv")

# Convert character to factor
load_reg_train[] <- lapply(
  load_reg_train,
  function(x){
    if(is.character(x)) as.factor(x) else x
  }
)

test[] <- lapply(
  test,
  function(x){
    if(is.character(x)) as.factor(x) else x
  }
)

# One-hot encoding
x_train <- model.matrix(
  happiness ~ .,
  data = load_reg_train
)[,-1]

y_train <- load_reg_train$happiness

x_test <- model.matrix(
  ~ .,
  data = test
)[,-1]

# Train
set.seed(123)

xgb_model <- xgboost(
  x = x_train,
  y = y_train,
  objective = "reg:squarederror",
  nrounds = 300,
  max_depth = 4,
  learning_rate = 0.05,
  subsample = 0.8,
  colsample_bytree = 0.8
)

# Predict
pred.label <- predict(
  xgb_model,
  x_test
)

# Export
write.csv(
  data.frame(
    RowIndex = seq_len(nrow(test)),
    Prediction = pred.label
  ),
  "RegressionPredictLabel.csv",
  row.names = FALSE
)